# fio Engine comparison


**Platform:** Aurora (HPE Cray EX), DAOS pool `e2sar`  
**Pattern:** random write (`randwrite`)  
**Config:** bs=1m, numjobs=16, iodepth=16, runtime=60 s, group_reporting  
**Configs tested:**

| # | Interface | fio engine | w/ IL (libioil) |
|---|-----------|-----------|----------------|
| 1 | DFS (native) | `dfs` | — |
| 2 | DFuse | `libaio` | No |
| 3 | DFuse | `psync` | No |
| 4 | DFuse + libioil | `libaio` | Yes |
| 5 | DFuse + libioil | `psync` | Yes |
| 6 | DFuse + libioil | `pvsync2` | Yes |

Duplicate runs are averaged. Runs with `elapsed < 60 s` are excluded.  
Any fio log lines emitted before the JSON are captured and listed as hiccups.

In [1]:
import json, re
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

RESULTS_DIR = Path(".")

# Config key: (fio_engine, has_il)
# Grouped by engine, IL-first within each engine so the table reads:
#   Engine → w/ libioil → DFuse (no IL)
CONFIG_ORDER = [
    ("dfs",     False),
    ("libaio",  True),
    ("libaio",  False),
    ("psync",   True),
    ("psync",   False),
    ("pvsync2", True),
]

CONFIG_LABELS = {          # short labels for chart x-axis
    ("dfs",     False): "DFS\n(native)",
    ("libaio",  True):  "libaio\n+IL",
    ("libaio",  False): "libaio\nDFuse",
    ("psync",   True):  "psync\n+IL",
    ("psync",   False): "psync\nDFuse",
    ("pvsync2", True):  "pvsync2\n+IL",
}

CONFIG_LABELS_FULL = {     # full labels for tables / legend
    ("dfs",     False): "DFS (native)",
    ("libaio",  True):  "DFuse + libioil + libaio",
    ("libaio",  False): "DFuse + libaio",
    ("psync",   True):  "DFuse + libioil + psync",
    ("psync",   False): "DFuse + psync",
    ("pvsync2", True):  "DFuse + libioil + pvsync2",
}

COLORS = {
    ("dfs",     False): "#1f77b4",
    ("libaio",  True):  "#aec7e8",
    ("libaio",  False): "#ff7f0e",
    ("psync",   True):  "#2ca02c",
    ("psync",   False): "#d62728",
    ("pvsync2", True):  "#98df8a",
}

HATCHES = {
    ("dfs",     False): "",
    ("libaio",  True):  "///",
    ("libaio",  False): "",
    ("psync",   True):  "///",
    ("psync",   False): "",
    ("pvsync2", True):  "///",
}

In [2]:
def load_fio_json(fpath: Path):
    """Read fio JSON, stripping leading non-JSON log lines. Returns (dict, pre_lines)."""
    content = fpath.read_text()
    idx = content.find('{')
    if idx == -1:
        return None, content.splitlines()
    pre_lines = [ln for ln in content[:idx].splitlines() if ln.strip()]
    try:
        return json.loads(content[idx:]), pre_lines
    except json.JSONDecodeError:
        return None, pre_lines


def config_key_from_stem(stem: str):
    """Map filename stem to (fio_engine, has_il). Returns None to skip.

    Filenames follow two patterns:
      fio_{interface}_{rest}          e.g. fio_dfs_*, fio_dfuse_*, fio_libioil_*
      fio_{interface}-{engine}_{rest} e.g. fio_dfuse-psync_*, fio_libioil-psync_*
    Engine separator can be '-' or '_'.
    """
    if "_dfs_" in stem:
        return ("dfs", False)
    has_il = "_libioil" in stem
    m = re.search(r'[-_](libaio|pvsync2|psync)_', stem)
    if m:
        return (m.group(1), has_il)
    if "_dfuse_" in stem:
        return ("libaio", False)
    if "_libioil_" in stem:
        return ("libaio", True)
    return None


_NUM_COLS = ["bw_GiBs", "iops", "lat_mean_ms", "clat_mean_ms", "slat_mean_us",
             "clat_p50_ms", "clat_p99_ms", "clat_p999_ms", "clat_p9999_ms",
             "usr_cpu", "sys_cpu"]

raw_rows = []
hiccups  = []   # (filename, [warning lines])

for fpath in sorted(RESULTS_DIR.glob("fio_*.json")):
    d, pre_lines = load_fio_json(fpath)
    if pre_lines:
        hiccups.append((fpath.name, pre_lines))
    if d is None:
        continue
    j = d["jobs"][0]
    w = j["write"]
    elapsed = j.get("elapsed", 0)
    if elapsed < 60:                         # skip runs shorter than the intended 60 s
        print(f"SKIP (elapsed={elapsed}s): {fpath.name}")
        continue
    key = config_key_from_stem(fpath.stem)
    if key is None:
        print(f"SKIP (no config match): {fpath.name}")
        continue
    pct = w["clat_ns"]["percentile"]
    raw_rows.append(dict(
        config_key    = key,
        file          = fpath.name,
        elapsed       = elapsed,
        bw_GiBs       = w["bw_bytes"] / 1024**3,
        iops          = w["iops"],
        lat_mean_ms   = w["lat_ns"]["mean"]  / 1e6,
        clat_mean_ms  = w["clat_ns"]["mean"] / 1e6,
        slat_mean_us  = w["slat_ns"]["mean"] / 1e3,
        clat_p50_ms   = pct["50.000000"]     / 1e6,
        clat_p99_ms   = pct["99.000000"]     / 1e6,
        clat_p999_ms  = pct["99.900000"]     / 1e6,
        clat_p9999_ms = pct["99.990000"]     / 1e6,
        usr_cpu       = j["usr_cpu"],
        sys_cpu       = j["sys_cpu"],
    ))

# --- show fio hiccup/warning messages captured before the JSON ---
print("\n=== fio log lines found before JSON output ===")
for fname, lines in hiccups:
    unique = sorted(set(lines))
    print(f"\n  {fname}")
    for ln in unique:
        print(f"    • {ln}")

raw_df = pd.DataFrame(raw_rows)
print(f"\nLoaded {len(raw_df)} valid runs across {raw_df['config_key'].nunique()} configs")

# Average over duplicate runs for the same config
df = (raw_df.groupby("config_key", sort=False)[_NUM_COLS].mean().reset_index())
df["label"]      = df["config_key"].map(CONFIG_LABELS)
df["label_full"] = df["config_key"].map(CONFIG_LABELS_FULL)
df["has_il"]     = df["config_key"].apply(lambda k: k[1])
df["ioengine"]   = df["config_key"].apply(lambda k: k[0])
df["color"]      = df["config_key"].map(COLORS)
df["hatch"]      = df["config_key"].map(HATCHES)

order_idx = {k: i for i, k in enumerate(CONFIG_ORDER)}
df["_ord"] = df["config_key"].map(order_idx)
df = df.sort_values("_ord").drop(columns="_ord").reset_index(drop=True)
df["n_runs"] = df["config_key"].map(raw_df.groupby("config_key").size().to_dict())


=== fio log lines found before JSON output ===

  fio_dfuse-psync_bs1m_nj16_iod16_1781751068.json
    • note: both iodepth >= 1 and synchronous I/O engine are selected, queue depth will be capped at 1

  fio_libioil-psync_bs1m_nj16_iod16_1781750925.json
    • note: both iodepth >= 1 and synchronous I/O engine are selected, queue depth will be capped at 1

  fio_libioil-pvsync2_bs1m_nj16_iod16_1781750842.json
    • note: both iodepth >= 1 and synchronous I/O engine are selected, queue depth will be capped at 1

Loaded 6 valid runs across 6 configs


In [3]:
# --- Summary comparison table — grouped by engine, IL before DFuse ---
engine_label = {"dfs": "DFS", "libaio": "libaio", "psync": "psync", "pvsync2": "pvsync2"}

rows_tbl = []
for _, row in df.iterrows():
    eng  = row["ioengine"]
    has_il = row["has_il"]
    iface = ("native" if eng == "dfs"
             else ("w/ libioil" if has_il else "DFuse (no IL)"))
    rows_tbl.append({
        "Engine":          engine_label[eng],
        "Interface":       iface,
        "Runs":            int(row["n_runs"]),
        "BW (GiB/s)":     round(row["bw_GiBs"],      2),
        "IOPS":            int(round(row["iops"])),
        "Mean Lat (ms)":   round(row["lat_mean_ms"],   2),
        "slat (µs)":       round(row["slat_mean_us"],  1),
        "usr CPU%":        round(row["usr_cpu"],        1),
        "sys CPU%":        round(row["sys_cpu"],        1),
        "Total CPU%":      round(row["usr_cpu"] + row["sys_cpu"], 1),
    })

tbl = pd.DataFrame(rows_tbl).set_index(["Engine", "Interface"])

tbl.style \
   .set_caption("fio randwrite bs=1m nj=16 iodepth=16 — Aurora DAOS  (duplicate runs averaged)") \
   .format({
       "BW (GiB/s)":   "{:.2f}",
       "IOPS":         "{:,}",
       "Mean Lat (ms)":"{:.2f}",
       "slat (µs)":    "{:.0f}",
       "usr CPU%":     "{:.1f}",
       "sys CPU%":     "{:.1f}",
       "Total CPU%":   "{:.1f}",
   }) \
   .highlight_max(subset=["BW (GiB/s)", "IOPS"],    color="lightblue") \
   .highlight_min(subset=["Mean Lat (ms)"],           color="lightgreen")

## Key Findings

**Setup:** fio random write, bs=1m, numjobs=16, iodepth=16, runtime=60 s, group_reporting.  
Aurora DAOS pool `e2sar`. Results collected 2026-06-17. Runs with `elapsed < 60 s` excluded.

---

### Summary table

| Engine | Interface | BW (GiB/s) | IOPS | Mean Lat (ms) | slat (µs) | Total CPU% |
|--------|-----------|-----------|------|--------------|-----------|-----------|
| DFS    | native        | **89.5** | **91,623** | 2.65  | 14.7  | 97.0 |
| libaio | w/ libioil    | 14.3     | 14,656     | 17.15 | 838.8 | 23.8 |
| libaio | DFuse (no IL) | 14.7     | 15,024     | 16.75 | 785.8 | 27.2 |
| psync  | w/ libioil    | **32.9** | **33,692** | **0.41** | ≈0 | 99.8 |
| psync  | DFuse (no IL) | 13.3     | 13,600     | 1.02  | ≈0    | 14.0 |
| pvsync2| w/ libioil    | 13.6     | 13,888     | 1.03  | ≈0    | 11.5 |

---

### 1. DFS (native) is 6× faster than any POSIX-path engine
The DAOS native `dfs` ioengine bypasses FUSE entirely, issuing RDMA writes directly from userspace. With 256 in-flight IOs (nj=16 × iodepth=16) it delivers **89.5 GiB/s** and **91,623 IOPS** — ~6× the throughput of the best FUSE-path result. Cost: ~97% CPU per node due to active polling.

### 2. libioil + psync delivers 2.5× speedup over DFuse + psync
**32.9 GiB/s vs 13.3 GiB/s** (2.5× gain, single clean run).  
`psync` issues `pwrite()` per IO → `libioil` intercepts it → data routes directly to DAOS, bypassing the FUSE kernel context switch entirely. Mean latency drops from 1.02 ms → 0.41 ms (2.5×). CPU cost spikes to ~99.8% (DAOS userspace polling replaces the FUSE wait).

### 3. libioil + libaio has no effect — AIO syscalls are not intercepted
**14.3 GiB/s with IL vs 14.7 GiB/s without** (−3%).  
fio's `libaio` engine issues `io_submit`/`io_getevents` kernel AIO syscalls. `libioil` has no hooks for these — data still travels through FUSE unchanged. The slight regression is libioil's per-fd tracking overhead on every `open()`/`close()` with no offsetting benefit.

### 4. libioil + pvsync2 shows no speedup over DFuse + psync
**13.6 GiB/s vs 13.3 GiB/s** (≈ equal, within noise).  
`pvsync2` uses `pwritev2()` with extra flags (RWF\_HIPRI, etc.). IL intercept coverage for `pwritev2` may be incomplete in the tested libioil version, or the per-call overhead negates the FUSE bypass benefit. Very low CPU (11.5%) despite IL — consistent with data still going through FUSE.

### 5. psync mean latency is 16× lower than libaio on DFuse (no IL)
DFuse + psync: **1.02 ms** mean lat vs DFuse + libaio: **16.75 ms** (16× difference).  
With `psync`, iodepth is capped to 1 per job — no queue buildup, so lat\_mean equals the raw FUSE round-trip (~1 ms). With `libaio` (iodepth=16), 256 IOs share a deep queue and lat includes queue wait time, inflating the mean ~16×.

### 6. DFuse submission latency (slat) is the IOPS ceiling for async engines
DFuse + libaio slat = **785.8 µs** vs DFS = **14.7 µs** (53×).  
Each IO through the FUSE kernel module incurs a context switch into the DFuse daemon and back. This per-IO overhead caps async IOPS at ~15 K regardless of iodepth — adding more queue depth cannot overcome this bottleneck.

### 7. DFS tail latency is wide despite the low mean
DFuse clat is narrow and consistent: p50 = 15.9 ms, p99.99 = 21.9 ms (6 ms spread).  
DFS is faster on average (p50 = 2.5 ms) but has a long tail: p99.99 = 51.1 ms (49 ms spread). Occasional outliers likely reflect DAOS client retransmission or fabric jitter under 256-IO concurrency.

---

## Data quality — bugs and hiccups per config

| Config | Status | Hiccups / Notes |
|--------|--------|-----------------|
| **DFS (native)** | ✅ Clean | None. |
| **DFuse + libaio** | ✅ Clean | None. File renamed to `fio_dfuse_libaio_*` for clarity. |
| **DFuse + psync** | ✅ Clean | fio emits `note: both iodepth >= 1 and synchronous I/O engine are selected, queue depth will be capped at 1` to stdout before the JSON. Informational only — expected behavior when pairing `iodepth=16` with a sync engine. A separate run (timestamp 1781751047) terminated early after ~14 s and was discarded. |
| **DFuse+libioil + libaio** | ✅ Clean | File renamed to `fio_libioil_libaio_*` for clarity. An earlier run (`fio_libioil-libaio_..._1781750781`) was terminated after 14 s (`fio: terminating on signal 2`) and deleted. |
| **DFuse+libioil + psync** | ✅ Clean | fio emits the same iodepth cap note as above (informational). An earlier run (timestamp 1781750624, BW = 21.7 GiB/s) was removed by user; only the higher-BW run (32.9 GiB/s) remains. |
| **DFuse+libioil + pvsync2** | ✅ Clean | fio emits the iodepth cap note (informational). |